# FilmAura — Hybrid Cinema Recommender System
### Machine Learning Essentials (Sem 5 Capstone)

**Objective**: Build a multi-modal hybrid recommender system combining:
1. **Content-Based Filtering**: TF-IDF n-gram vectorization & Cosine Similarity on synopsis, keywords, director, cinematographer, and cast.
2. **10-Dimensional Mood Resonance**: Mathematical mapping across emotional frequencies.
3. **Bayesian Shrinkage Rating Weighting**: Minimum-vote threshold stabilization formula ($WR = \frac{v}{v+m}R + \frac{m}{v+m}C$).
4. **Aura Diagnostic Transparency**: Decomposing recommendation rationale into interpretable components.

In [ ]:
import sys
import os
import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Add project src to path
sys.path.append(os.path.abspath('..'))
from src.data_loader import get_movies_dataframe

print("Libraries loaded successfully.")

## 1. Exploratory Data Analysis & Catalogue Inspection

In [ ]:
df = get_movies_dataframe()
print(f"Total films in catalogue: {len(df)}")
display_cols = ['title', 'year', 'runtime', 'rating', 'votes', 'director']
df[display_cols].head(10)

## 2. Text Corpus Construction & TF-IDF Vectorization

In [ ]:
def build_corpus(row):
    genres = " ".join(row["genres"])
    cast = " ".join([c["name"] for c in row["cast"]])
    return f"{row['title']} {genres} {row['director']} {row['cinematographer']} {cast} {row['synopsis']} {row['keywords']}"

df["corpus"] = df.apply(build_corpus, axis=1)
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df["corpus"])
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

## 3. Cosine Similarity Matrix Computation

In [ ]:
cos_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Pairwise Cosine Matrix: {cos_sim.shape}")

# Check sample similarity with Blade Runner 2049
br_idx = df[df['title'] == 'Blade Runner 2049'].index[0]
sim_scores = list(enumerate(cos_sim[br_idx]))
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:6]

print("Top 5 Content-Similar Films to Blade Runner 2049:")
for idx, score in sim_scores:
    print(f"- {df.iloc[idx]['title']}: {score:.4f}")

## 4. 10-Dimensional Mood Resonance Scoring & Bayesian Shrinkage

In [ ]:
# Bayesian parameters
C = df['rating'].mean()
m = df['votes'].quantile(0.35)

def bayesian_rating(v, r):
    return (v / (v + m)) * r + (m / (v + m)) * C

df['bayesian_rating'] = df.apply(lambda r: bayesian_rating(r['votes'], r['rating']), axis=1)
df[['title', 'rating', 'votes', 'bayesian_rating']].sort_values(by='bayesian_rating', reverse=True).head(5)

## 5. End-to-End Hybrid Re-Ranking & Model Export

In [ ]:
from src.recommender import FilmAuraRecommender
rec = FilmAuraRecommender()
results = rec.recommend(primary_mood="Dark", secondary_mood="Suspenseful", anchor_movie_ids=["blade-runner-2049-2017"], top_n=5)

print("\n--- FilmAura Live Calibration Output ---")
for i, r in enumerate(results):
    print(f"#{i+1} {r['title']} ({r['year']}) — Aura Match: {r['aura_match_pct']}%")
    print(f"   Rationale: {r['rationale']}")
    print(f"   Stream: {r['stream_provider']}\n")